# 🚀 Hardware-Accelerated WebGPU Google Chrome on Google Colab (Tesla T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hosein-ul/colab-webgpu-chrome/blob/main/colab_chrome_webgpu.ipynb)

این نوت‌بوک محیط سرور لینوکس ابری Google Colab را با کارت گرافیک **NVIDIA Tesla T4** و شتاب کامل سخت‌افزاری **WebGPU** پیکربندی کرده و دو روش استریم ریموت در اختیار شما قرار می‌دهد:

* **روش اول (پیشنهادی و ۱۰۰٪ تک‌کلیک):** استریم فوری داخل مرورگر با noVNC بهینه‌شده و بدون نیاز به هیچ اکانت، لاگین یا کپی کردن کد.
* **روش دوم (فوق‌سریع ۶۰ فریم WebRTC):** استریم با Google Chrome Remote Desktop با تاخیر زیر ۳۰ میلی‌ثانیه برای نهایت کیفیت و روانی.

> ⚠️ **مهم / Important:** مطمئن شوید از منوی بالا گزینه **Runtime ➔ Change runtime type** روی **T4 GPU** تنظیم شده باشد.

In [ ]:
#@title 🟢 [روش اول - پیشنهادی] راه‌اندازی ۱۰۰٪ تک‌کلیک مرورگر وب (1-Click Instant Stream)
#@markdown ### ⚡ با زدن دکمه Play، همه‌چیز (درایورهای T4، ولکان، کروم WebGPU و لینک تونل) به صورت کاملاً خودکار در کمتر از ۴۵ ثانیه آماده می‌شود:

TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1366x768", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re
from IPython.display import display, HTML

print("⏳ [1/5] پیکربندی درایورهای سطح هسته NVIDIA Tesla T4 و Vulkan...")
os.makedirs("/dev/dri", exist_ok=True)
subprocess.run("mknod -m 666 /dev/dri/card0 c 226 0 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/dri/renderD128 c 226 128 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/nvidia-modeset c 195 254 > /dev/null 2>&1", shell=True)
subprocess.run("chmod -R 666 /dev/dri /dev/nvidia* > /dev/null 2>&1", shell=True)

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd_content = '{\n    "file_format_version": "1.0.0",\n    "ICD": {\n        "library_path": "libGLX_nvidia.so.0",\n        "api_version": "1.3.275"\n    }\n}'
for p in ["/etc/vulkan/icd.d/nvidia_icd.json", "/usr/share/vulkan/icd.d/nvidia_icd.json"]:
    with open(p, "w") as f: f.write(icd_content)

with open("/etc/ld.so.conf.d/nvidia.conf", "w") as f:
    f.write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

print("⏳ [2/5] نصب سریع ابزارهای بهینه‌شده نمایش تصویر...")
subprocess.run("apt-get update -qq && apt-get install -y -qq xvfb fluxbox x11vnc websockify novnc autocutsel > /dev/null 2>&1", shell=True)

if not os.path.exists("/usr/bin/google-chrome"):
    print("⏳ [3/5] دانلود و نصب Google Chrome Stable...")
    chrome_deb = "/tmp/chrome.deb"
    subprocess.run(f"wget -q -O {chrome_deb} https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb", shell=True)
    subprocess.run(f"dpkg -i {chrome_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
    subprocess.run(f"rm -f {chrome_deb}", shell=True)
else:
    print("⚡ [3/5] کروم از قبل نصب است.")

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ [4/5] دانلود تونل Cloudflare...")
    subprocess.run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True)

print("⏳ [5/5] راه‌اندازی سرور تصویر Low-Latency و اجرای کروم با WebGPU...")
subprocess.run("pkill -9 -f 'Xvfb|x11vnc|websockify|fluxbox|cloudflared|google-chrome'", shell=True)
time.sleep(1)

env = os.environ.copy()
env["DISPLAY"] = ":1"
env["VK_ICD_FILENAMES"] = "/etc/vulkan/icd.d/nvidia_icd.json"

subprocess.Popen(["Xvfb", ":1", "-screen", "0", f"{RESOLUTION}x24"], env=env)
time.sleep(1)
subprocess.Popen(["fluxbox"], env=env)
subprocess.Popen(["autocutsel", "-fork", "-s", "CLIPBOARD"], env=env)
subprocess.Popen(["autocutsel", "-fork", "-s", "PRIMARY"], env=env)

# تنظیمات low-latency برای حذف لگ: بدون ncache برای کاهش فشار رم، رندر موس سمت کلاینت و تاخیر صفر
vnc_cmd = [
    "x11vnc",
    "-display", ":1",
    "-nopw",
    "-listen", "127.0.0.1",
    "-forever",
    "-rfbport", "5900",
    "-wait", "0",
    "-defer", "0",
    "-threads",
    "-nowf",
    "-noxrecord",
    "-cursor", "arrow",
    "-ncache", "0"
]
subprocess.Popen(vnc_cmd, env=env)
subprocess.Popen(["websockify", "--web", "/usr/share/novnc", "6080", "localhost:5900"], env=env)
time.sleep(1)

chrome_cmd = [
    "google-chrome",
    "--no-sandbox",
    "--disable-gpu-sandbox",
    "--enable-unsafe-webgpu",
    "--enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService",
    "--use-vulkan=native",
    "--use-angle=vulkan",
    "--ignore-gpu-blocklist",
    "--disable-dev-shm-usage",
    "--enable-gpu-rasterization",
    "--enable-zero-copy",
    "--disable-background-timer-throttling",
    "--disable-backgrounding-occluded-windows",
    "--disable-renderer-backgrounding",
    f"--window-size={RESOLUTION.split('x')[0]},{RESOLUTION.split('x')[1]}",
    "--start-maximized",
    "--user-data-dir=/tmp/chrome_colab_profile",
    TARGET_URL
]
subprocess.Popen(chrome_cmd, env=env)
time.sleep(2)

log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file): os.remove(log_file)
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:6080", "--logfile", log_file])

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    link = f"{tunnel_url}/vnc.html?autoconnect=true&resize=scale&quality=7&compression=2"
    print("\n" + "═"*60)
    print("🎉 همه‌چیز آماده است! (NVIDIA Tesla T4 + Hardware WebGPU)")
    print("═"*60)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #1b5e20 0%, #2e7d32 100%); padding: 22px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.25); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #b9f6ca;">⚡ مرورگر ریموت آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                کروم با شتاب کامل کارت گرافیک تسلا T4 و لینک ماینینگ UniCred لود شد. برای ورود روی دکمه زیر کلیک کنید:
            </p>
            <a href="{link}" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به مرورگر کروم ریموت
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                لینک مستقیم: <a href="{link}" target="_blank" style="color: #b9f6ca;">{link}</a>
            </p>
        </div>
    '''))
else:
    print("❌ خطا در دریافت آدرس تونل. لطفاً دوباره سلول را اجرا کنید.")


In [ ]:
#@title ⚡ [روش دوم - پیشرفته ۶۰ فریم] راه‌اندازی با Google Chrome Remote Desktop (WebRTC)
#@markdown ### 📋 راهنمای اتصال ریموت دسکتاپ گوگل (تاخیر زیر ۳۰ میلی‌ثانیه و ۶۰ فریم):
#@markdown 1. در یک تب جدید به **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** بروید.
#@markdown 2. دکمه **Begin** ➔ **Next** ➔ **Authorize** را بزنید و کد Debian Linux را کپی کنید.
#@markdown 3. کد را در فیلد `AUTH_COMMAND` زیر پیست کرده و دکمه **Play** را بزنید:

AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}

import os, sys, time, subprocess, re
from IPython.display import display, HTML

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 دریافت کد لینوکس: https://remotedesktop.google.com/headless")
    raw_auth = input("دستور احراز هویت را وارد کنید: ").strip()

if not raw_auth:
    raise ValueError("دستور احراز هویت الزامی است.")

print("⏳ [1/5] پیکربندی درایورهای سطح هسته NVIDIA Tesla T4 و Vulkan...")
os.makedirs("/dev/dri", exist_ok=True)
subprocess.run("mknod -m 666 /dev/dri/card0 c 226 0 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/dri/renderD128 c 226 128 > /dev/null 2>&1", shell=True)
subprocess.run("mknod -m 666 /dev/nvidia-modeset c 195 254 > /dev/null 2>&1", shell=True)
subprocess.run("chmod -R 666 /dev/dri /dev/nvidia* > /dev/null 2>&1", shell=True)

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd_content = '{\n    "file_format_version": "1.0.0",\n    "ICD": {\n        "library_path": "libGLX_nvidia.so.0",\n        "api_version": "1.3.275"\n    }\n}'
for p in ["/etc/vulkan/icd.d/nvidia_icd.json", "/usr/share/vulkan/icd.d/nvidia_icd.json"]:
    with open(p, "w") as f: f.write(icd_content)

with open("/etc/ld.so.conf.d/nvidia.conf", "w") as f:
    f.write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

print("⏳ [2/5] ایجاد کاربر اختصاصی و تنظیم مجوزهای دسترسی...")
os.makedirs("/home/colab", exist_ok=True)
subprocess.run("id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab >/dev/null 2>&1", shell=True)
subprocess.run("echo 'colab:123456' | chpasswd", shell=True)
subprocess.run("usermod -aG sudo,video,render colab >/dev/null 2>&1", shell=True)
with open("/etc/sudoers.d/colab", "w") as f:
    f.write("colab ALL=(ALL) NOPASSWD:ALL\n")

print("⏳ [3/5] به‌روزرسانی پکیج‌ها و نصب XFCE4 و Chrome Remote Desktop...")
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get update -qq", shell=True)
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal xscreensaver > /dev/null 2>&1", shell=True)
subprocess.run("systemctl disable lightdm.service > /dev/null 2>&1 || true", shell=True)

crd_deb = "/tmp/crd.deb"
subprocess.run(f"wget -q -O {crd_deb} https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb", shell=True)
subprocess.run(f"dpkg --install {crd_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
subprocess.run(f"rm -f {crd_deb}", shell=True)
subprocess.run("usermod -aG chrome-remote-desktop colab >/dev/null 2>&1", shell=True)

crd_session = "exec /etc/X11/Xsession /usr/bin/xfce4-session\n"
with open("/etc/chrome-remote-desktop-session", "w") as f: f.write(crd_session)
with open("/home/colab/.chrome-remote-desktop-session", "w") as f: f.write(crd_session)

print("⏳ [4/5] نصب Google Chrome و تنظیم اجرای خودکار WebGPU...")
if not os.path.exists("/usr/bin/google-chrome"):
    chrome_deb = "/tmp/chrome.deb"
    subprocess.run(f"wget -q -O {chrome_deb} https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb", shell=True)
    subprocess.run(f"dpkg -i {chrome_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
    subprocess.run(f"rm -f {chrome_deb}", shell=True)

launch_script = f"""#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-vulkan=native \\
  --use-angle=vulkan \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --disable-background-timer-throttling \\
  --disable-backgrounding-occluded-windows \\
  --disable-renderer-backgrounding \\
  --start-maximized \\
  "$@" \\
  "{TARGET_URL}" &
"""
with open("/home/colab/launch_chrome.sh", "w") as f: f.write(launch_script)
subprocess.run("chmod +x /home/colab/launch_chrome.sh", shell=True)

os.makedirs("/home/colab/.config/autostart", exist_ok=True)
os.makedirs("/home/colab/Desktop", exist_ok=True)
desktop_entry = """[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Comment=Google Chrome with Hardware WebGPU on Tesla T4
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Path=/home/colab
Terminal=false
StartupNotify=true
"""
with open("/home/colab/.config/autostart/webgpu-chrome.desktop", "w") as f: f.write(desktop_entry)
with open("/home/colab/Desktop/WebGPU_Chrome.desktop", "w") as f: f.write(desktop_entry)
subprocess.run("chmod +x /home/colab/Desktop/WebGPU_Chrome.desktop", shell=True)
subprocess.run("chown -R colab:colab /home/colab && chmod +x /home/colab/.chrome-remote-desktop-session", shell=True)

print("⏳ [5/5] راه‌اندازی هاست Chrome Remote Desktop با اکانت شما...")
if raw_auth.startswith("4/"):
    exec_cmd = f'/opt/google/chrome-remote-desktop/start-host --code="{raw_auth}" --redirect-url="https://remotedesktop.google.com/_/oauthredirect" --name=colab-t4'
else:
    exec_cmd = raw_auth

pin_str = str(PIN)
if "--pin=" not in exec_cmd:
    exec_cmd += f" --pin={pin_str}"

exec_cmd = re.sub(r'^DISPLAY=\s*', '', exec_cmd)
full_cmd = f'su - colab -c \'{exec_cmd}\''

proc = subprocess.Popen(
    full_cmd,
    shell=True,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
try:
    stdout, stderr = proc.communicate(input=f"{pin_str}\n{pin_str}\n", timeout=25)
except subprocess.TimeoutExpired:
    proc.kill()
    stdout, stderr = proc.communicate()

time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop", shell=True, capture_output=True, text=True)

print("\n" + "═"*60)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Chrome Remote Desktop با موفقیت راه‌اندازی شد!")
    print(f"🔑 پین ورود شما: {pin_str}")
    print("═"*60)
    display(HTML(f'''
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری آماده اتصال است! (WebRTC 60 FPS)</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                سیستم <b>colab-t4</b> در پنل گوگل اضافه شد. روی دکمه زیر کلیک کرده و پین <b>{pin_str}</b> را وارد نمایید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    '''))
else:
    print("⚠️ پیام خروجی:")
    print(stdout)
    if stderr: print("خطاها:", stderr)
    print("═"*60)


In [ ]:
#@title 📊 مانیتور زنده هش‌ریت و مصرف کارت گرافیک تسلا T4 (Real-Time GPU Monitor)
#@markdown وضعیت زنده درصد درگیری GPU، حافظه VRAM، توان مصرفی و دمای کارت گرافیک:

import time, subprocess
from IPython.display import clear_output

print("Starting lightweight GPU monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*60)
            print(f"🚀 UNICRED WEBGPU MINING MONITOR — {name}")
            print("="*60)
            print(f"  GPU Compute Utilization : {gpu_util}%")
            print(f"  VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  GPU Temperature         : {temp} °C")
            print(f"  Power Consumption       : {power} W")
            print("="*60)
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
